In [1]:
import numpy as np
import matplotlib.pyplot as plt
# --------------------
# 1. 파라미터 설정
# --------------------
M = 100            # 최대 클러스터 크기
Q = 0.5
s_co = 2.5
s_cm = 0.5
gamma_mo = 12.8
gamma_cm = 2.6
gamma_co = 15.4
f0 = 1e6
g0 = 2e7
L = g0 / f0        # = 200
dx = 1e-4          # 시간 스텝 (무차원)
t_micro = 10e-6    # 10 μs
x_max = f0 * t_micro
steps = int(x_max / dx)

# --------------------
# 2. 계수 계산
# --------------------
a = np.zeros((M, M))
b = np.zeros((M, M))
c = np.zeros((M, M))
d = np.zeros((M, M))
e = np.zeros((M, M))
h = np.zeros((M, M))

for i in range(2, M):
    for n in range(1, i + 1):
        if n < i:
            a[i, n] = (i - 1)**(2/3) * np.exp(s_cm + gamma_mo * (i**(2/3) - (i - 1)**(2/3)))
        b[i, n] = i**(2/3) * np.exp(s_co)
        if n == i:
            b[i, n] *= (1 - Q)
        if n >= 2:
            c[i, n] = L * (n - 1)**(2/3) * np.exp(gamma_cm * (n**(2/3) - (n - 1)**(2/3)))
        if n < i:
            d[i, n] = L * n**(2/3) * np.exp(s_cm)
        if n == i:
            e[i, n] = Q * (i - 1)**(2/3) * np.exp(gamma_co * (i**(2/3) - (i - 1)**(2/3)))
            h[i, n] = Q * i**(2/3) * np.exp(s_co)

# --------------------
# 3. 초기 조건 설정
# --------------------
F = np.zeros((M, M))
F[2, 1] = 1.0  # 초기 seed cluster

# --------------------
# 4. 미분 함수 정의
# --------------------
def compute_dF(F):
    dF = np.zeros_like(F)
    for i in range(2, M):
        for n in range(1, i + 1):
            term = 0.0
            if i > 2:
                term += a[i, n] * (F[i - 1, n] - F[i, n])
            if i < M - 1:
                term -= b[i, n] * (F[i, n] - F[i + 1, n])
            if n > 1:
                term += c[i, n] * (F[i, n - 1] - F[i, n])
            if n < i:
                term -= d[i, n] * (F[i, n] - F[i, n + 1])
            if n == i and i > 2:
                term += e[i, n] * (F[i - 1, i - 1] - F[i, i])
            if n == i and i < M - 1:
                term -= h[i, n] * (F[i, i] - F[i + 1, i + 1])
            dF[i, n] = term
    return dF

# --------------------
# 5. RK4 적분
# --------------------
for step in range(steps):
    k1 = compute_dF(F)
    k2 = compute_dF(F + 0.5 * dx * k1)
    k3 = compute_dF(F + 0.5 * dx * k2)
    k4 = compute_dF(F + dx * k3)
    F += (dx / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

# --------------------
# 6. 결과 F[i,n] → numpy array (M x M)
# --------------------
# F는 시간 x = f0 * 10us 까지 통합한 최종 결과
# --------------------
# 7. Plot F[i,n] as heatmap
# --------------------
fig, ax = plt.subplots(figsize=(8, 6))

# i: 2 ~ M-1, n: 1 ~ i
I, N = np.meshgrid(np.arange(M), np.arange(M), indexing='ij')
valid_mask = (I >= 2) & (N >= 1) & (N <= I)
F_masked = np.ma.masked_where(~valid_mask, F)

cmap = ax.imshow(F_masked, origin='lower', aspect='auto',
                 extent=[0, M, 0, M], cmap='viridis')
ax.set_title('Cluster Distribution $F_{i,n}$ at t = 10 μs')
ax.set_xlabel('Cluster size i')
ax.set_ylabel('Number of C-phase monomers n')
fig.colorbar(cmap, label='$F_{i,n}$')

plt.tight_layout()
plt.show()


KeyboardInterrupt: 

In [ ]:
import numpy as np
#기본 parameter
k_B = 1.38e-23
T = 230
c_co = (36*np.pi)^(1/3)
dt = 1e-4
#입력받을 parameter
volume_c, volume_m, volume_o = 
chemical_potential_c, chemical_potential_m, chemical_potential_o = 
surface_E_co, surface_E_cm, surface_E_mo = 
C1 =
M = 
#자동 계산 되는 parameter
delta_chemical_co = chemical_potential_o - chemical_potential_c
delta_chemical_cm = chemical_potential_m - chemical_potential_c
delta_chemical_mo = chemical_potential_o - chemical_potential_m
supersatuered_co = delta_chemical_co/(k_B*T)
supersatuered_cm = delta_chemical_cm/(k_B*T)
supersatuered_mo = delta_chemical_mo/(k_B*T)
interface_co = c_co*(volume_c^(2/3))*surface_E_co/(k_B*T)
interface_cm = c_co*(volume_c^(2/3))*surface_E_cm/(k_B*T)
interface_mo = c_co*(volume_m^(2/3))*surface_E_mo/(k_B*T)

#Work of formation
w(i,n) = -supersatuered_mo*i + interface_mo*(i+(volume_c/volume_m - 1)*n)^(2/3) -supersatuered_cm*n + interface_cm * n^(2/3)

#critcal values
i_star = (2*interface_mo/(3*supersatuered_mo))^(3)
n_star = (2*interface_cm/(3*supersatuered_cm))^(3)
i_co_star = (2*interface_co/(3*supersatuered_co))^(3)

#equilibrium concentration
C(i,n) = C1*exp(w(1,1)-w(i,n))

#intial distribution
Z(1,1) = C1

#master equation
Z(i,n)/dt = I(i-1,n) - I(i,n) + G(i,n-1) - G(i,n) (for M-1>i>3 , i-1 > n >2)
Z(i,1)/dt = I(i-1,1) - I(i,1) + G(i,1) (for M-1 > i > 2 , n = 1)
Z(i,i)/dt = K(i-1,i-1) -K(i,i) +G(i,i-1) - I(i,i) (for M-1>i>2 , n = i)

#frequecncy equation
I(i,n) = f(i,n)*C(i,n)*(Z(i,n)/C(i,n)-Z(i+1,n)/C(i+1,n))
G(i,n) = g(i,n)*C(i,n)*(Z(i,n)/C(i,n)-Z(i,n+1)/C(i,n+1))
K(i,n) = k(i,n)*C(i,n)*(Z(i,n)/C(i,n)-Z(i+1,n+1)/C(i+1,n+1))

#attachment rate detail
f(i,i) = (1-Q)*f_0*exp(supersatuered_co)*i^(2/3)
f(i,n) = f_0*exp(supersatuered_co)*i^(2/3)  (for n<i)
g(i,n) = g_0*exp(supersatuered_cm)*n^(2/3) (for n<i)
k(i,i) = Q*f_0*exp(supersatuered_co)*i^(2/3)

